## Описание проекта
Этот DAG должен ежедневно:

* проверять наличие новых файлов с данными;

* запускать Spark-задачу;

* формировать обновлённую итоговую таблицу.

Эта таблица станет основой для финансовых отчётов и аналитических дашбордов.

## Описание данных

Таблица `taxi_data` содержит данные об активности пользователей и состоит из следующих полей:

* `taxi_id` — идентификатор водителя;

* `trip_start_timestamp` — время начала поездки;

* `trip_end_timestamp` — время окончания поездки;

* `trip_seconds` — длительность поездки в секундах;

* `trip_miles` — дистанция поездки;

* `fare` — стоимость поездки;

* `tips` — размер чаевых;

* `trip_total` — общая стоимость поездки: стоимость поездки + чаевые + комиссия;

* `payment_type` — способ оплаты.

## Шаг 1. Настройте Spark-агрегацию

In [ ]:
# filename=my_spark_job.py
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys
from datetime import datetime

# Создаём Spark-сессию и при необходимости добавляем конфигурации
spark = SparkSession.builder.appName('*').config('*', '*').getOrCreate()

# Указываем порт и параметры кластера ClickHouse
jdbcPort = '*'
jdbcHostname = '*'
username = '*'
password = '*'
jdbcDatabase = 'playground_' + username
jdbcUrl = f'jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true'


# Считываем исходные данные за нужную дату
df = spark.read.parquet('*/taxi_data.parquet', inferSchema=True, header=True)

date_arg = sys.argv[1] 

dt = datetime.fromisoformat(date_arg)
clean_datetime_str = dt.strftime("%Y-%m-%d %H:%M:%S")

# Строим агрегат по пользователям
result_df = (df.groupBy('payment_type') # группируем по типу оплаты
    .agg(
        F.count('taxi_id').alias('taxi_count'), # количество поездок
        F.avg('fare').alias('avg_fare'), # средняя стоимость поездки
        F.avg('tips').alias('avg_tips'), # средние чаевые 
        F.sum('trip_total').alias('total_revenue') # общий доход 
    ).withColumn('date', F.to_timestamp(F.lit(clean_datetime_str), 'yyyy-MM-dd HH:mm:ss')) # для различия итоговых данных в таблице, я беру дату запуска скрипта и записываю в новый столбец.
) 

result_df.write.format('jdbc') \
    .option('url', jdbcUrl) \
    .option('user', username) \
    .option('password', password) \
    .option('dbtable', 'taxi_payment_summary') \
    .mode('append') \
    .save()


spark.stop()

## Шаг 2. Настройте DAG

In [ ]:
# filename=my_spark_job.py

from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

class PysparkJobOperator(DataprocCreatePysparkJobOperator):
    template_fields = ("cluster_id", "args")

DAG_ID = "taxi_payment_summary"

with DAG(
    DAG_ID,
    schedule='@daily',
    start_date=datetime(2026, 6, 8),
    tags=["taxi_payment_summary_aggregate"],
    catchup=False
) as dag:
    
    user = '*'

    # 1) Ждём появления входного файла в S3
    wait_for_input = S3KeySensor(
        task_id='data_s3_sensor',
        poke_interval=60, 
        timeout=3600,
        bucket_name='*',
        bucket_key='*/taxi_data.parquet',
        mode='poke',
        aws_conn_id='s3',
        wildcard_match=False
    )

    # 2) Запускаем PySpark-задание на кластере Dataproc (оператор Яндекс Облака)
    run_pyspark = PysparkJobOperator(
        task_id="run_pyspark_job",
        name="daily_pyspark_job",
        cluster_id="*",
        args=["{{ ts }}"], # дату парсим в формате datetime
        main_python_file_uri=f"*/{user}/jobs/my_spark_job.py",
    ) 

    # 3)  Зависимости 
    wait_for_input >> run_pyspark

## Шаг 3. Запустите DAG с помощью Airflow UI
Задача запущена через веб-интерфейс, успехом запуска является записанная таблица в БД
```sql
    select *
    from taxi_payment_summary
```

|payment_type|taxi_count|avg_fare|avg_tips|total_revenue|date|
|------------|----------:|--------:|--------:|-------------:|----|
|Cash|1409697|11.194248598877067|0.003325194540318936|16971150.869999863|2026-06-09 03:00:00|
|Credit Card|1108190|15.869293343229563|3.477157157788922|23160130.960003342|2026-06-09 03:00:00|
|Dispute|1842|13.327953311617806|0.0|27052.29|2026-06-09 03:00:00|
|No Charge|12789|13.535953437670331|0.1256941524565911|184374.59000000003|2026-06-09 03:00:00|
|Pcard|878|9.833428246013668|0.18314350797266515|9175.55|2026-06-09 03:00:00|
|Prcard|968|11.172262396694215|0.25764462809917354|11513.149999999998|2026-06-09 03:00:00|
|Unknown|5180|11.79162804171493|0.35713595983005136|67725.07000000005|2026-06-09 03:00:00|
|Way2ride|3|3.5|0.7933333333333333|14.379999999999999|2026-06-09 03:00:00|
|Cash|1409697|11.194248598877067|0.003325194540318936|16971150.869999863|2026-06-09 17:50:32|
|Credit Card|1108190|15.869293343229563|3.477157157788922|23160130.960003342|2026-06-09 17:50:32|
|Dispute|1842|13.327953311617806|0.0|27052.29|2026-06-09 17:50:32|
|No Charge|12789|13.535953437670331|0.1256941524565911|184374.59000000003|2026-06-09 17:50:32|
|Pcard|878|9.833428246013668|0.18314350797266515|9175.55|2026-06-09 17:50:32|
|Prcard|968|11.172262396694215|0.25764462809917354|11513.149999999998|2026-06-09 17:50:32|
|Unknown|5180|11.79162804171493|0.35713595983005136|67725.07000000005|2026-06-09 17:50:32|
|Way2ride|3|3.5|0.7933333333333333|14.379999999999999|2026-06-09 17:50:32|
|Cash|1409697|11.194248598877067|0.003325194540318936|16971150.869999863|2026-06-09 18:12:17|
|Credit Card|1108190|15.869293343229563|3.477157157788922|23160130.960003342|2026-06-09 18:12:17|
|Dispute|1842|13.327953311617806|0.0|27052.29|2026-06-09 18:12:17|
|No Charge|12789|13.535953437670331|0.1256941524565911|184374.59000000003|2026-06-09 18:12:17|
|Pcard|878|9.833428246013668|0.18314350797266515|9175.55|2026-06-09 18:12:17|
|Prcard|968|11.172262396694215|0.25764462809917354|11513.149999999998|2026-06-09 18:12:17|
|Unknown|5180|11.79162804171493|0.35713595983005136|67725.07000000005|2026-06-09 18:12:17|
|Way2ride|3|3.5|0.7933333333333333|14.379999999999999|2026-06-09 18:12:17|
|Cash|1409697|11.194248598877067|0.003325194540318936|16971150.869999863|2026-06-10 22:05:42|
|Credit Card|1108190|15.869293343229563|3.477157157788922|23160130.960003342|2026-06-10 22:05:42|
|Dispute|1842|13.327953311617806|0.0|27052.29|2026-06-10 22:05:42|
|No Charge|12789|13.535953437670331|0.1256941524565911|184374.59000000003|2026-06-10 22:05:42|
|Pcard|878|9.833428246013668|0.18314350797266515|9175.55|2026-06-10 22:05:42|
|Prcard|968|11.172262396694215|0.25764462809917354|11513.149999999998|2026-06-10 22:05:42|
|Unknown|5180|11.79162804171493|0.35713595983005136|67725.07000000005|2026-06-10 22:05:42|
|Way2ride|3|3.5|0.7933333333333333|14.379999999999999|2026-06-10 22:05:42|
